# Eve-3-SABER-0.5B — Pre-tokenized Training

Run all 3 cells in order. Walk away.

1. **Setup**: git pull, install deps, login
2. **Pre-tokenize**: Download all datasets to disk, tokenize to .bin files (~1-2h)
3. **Train**: Reads from .bin files via mmap. Auto-resumes. ~1-2s/step.

If the kernel dies, just re-run Cell 3 — it resumes from checkpoint.
Pre-tokenized data persists on disk, no need to re-download.

In [ ]:
# ============================================================
# CELL 1: SETUP
# ============================================================
import os, subprocess

WORKSPACE = os.environ.get('EVE_WORKSPACE_ROOT', '/workspace')
REPO_DIR = f'{WORKSPACE}/Eve-0.5B-saber'
REPO_URL = 'https://github.com/anthony-maio/Eve-0.5B-saber.git'

# Clone or update repo
if os.path.exists(REPO_DIR):
    print('Updating repo...')
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

# Install deps (NO torch on RunPod — it ships pre-installed with CUDA)
subprocess.run(
    'pip install -q transformers accelerate datasets safetensors '
    'tokenizers wandb numpy tqdm huggingface_hub zstandard'.split(),
    check=True,
)

# HF login
hf_token = os.environ.get('HF_TOKEN', '')
if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print('HF: logged in')

# WandB login
wandb_key = os.environ.get('WANDB_API_KEY', '')
if wandb_key:
    import wandb
    wandb.login(key=wandb_key)
    print('WandB: logged in')

os.chdir(REPO_DIR)
print(f'\n✅ Ready in {REPO_DIR}')

In [ ]:
# ============================================================
# CELL 2: PRE-TOKENIZE ALL DATASETS (~1-2h, resumable)
# ============================================================
# Downloads full parquet files (parallel, fast), tokenizes to .bin.
# Falls back to streaming for huge datasets (fineweb-edu, dclm).
# Safe to re-run — skips already-completed sources.
# Output: /workspace/eve-saber-05b/tokens/*.bin (~120GB total)

!python pretokenize.py

In [ ]:
# ============================================================
# CELL 3: TRAIN (auto-resumes, reads from .bin files)
# ============================================================
# Detects .bin files automatically → mmap DataLoader (~1-2s/step)
# Falls back to streaming if .bin files are missing.
# Re-run this cell after kernel restarts — it resumes from checkpoint.
# Checkpoints every 500 steps to /workspace/eve-saber-05b/checkpoints/

!cd /workspace/Eve-0.5B-saber && python train_full.py

In [ ]:
# ============================================================
# CELL 4: UPLOAD CHECKPOINT TO HUGGINGFACE (run manually)
# ============================================================
import os, sys, torch, json, shutil
from pathlib import Path

os.chdir('/workspace/Eve-0.5B-saber')
sys.path.insert(0, '.')

from huggingface_hub import HfApi
from transformers import GPT2TokenizerFast
from configuration_saber import SABERConfig
from modeling_saber import SABERForCausalLM

CKPT_DIR = Path('/workspace/eve-saber-05b/checkpoints')
EXPORT_DIR = Path('/workspace/eve-saber-05b/hf-upload')
REPO_DIR = Path('/workspace/Eve-0.5B-saber')
HF_REPO = 'anthonym21/eve-3-0.5b-saber'

ckpt_path = Path((CKPT_DIR / 'latest.txt').read_text().strip())
with open(ckpt_path / 'meta.json') as f:
    meta = json.load(f)
print(f'Checkpoint: {ckpt_path.name} | Step {meta["step"]:,} | {meta["tokens_seen"]/1e9:.2f}B tokens')

cfg = SABERConfig(
    vocab_size=50257, d_model=1536, n_heads=12, head_dim=128,
    n_layers=20, d_ff=2164, max_position_embeddings=2048,
    d_exp=192, d_anchor=96, n_anchors=64,
    curiosity_coeff=0.01, tie_word_embeddings=True,
)
mdl = SABERForCausalLM(cfg)
mdl.load_state_dict(torch.load(ckpt_path / 'model.pt', map_location='cpu'))
tok = GPT2TokenizerFast.from_pretrained('gpt2')
tok.pad_token = tok.eos_token

if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True)
mdl.save_pretrained(EXPORT_DIR)
tok.save_pretrained(EXPORT_DIR)
cfg.save_pretrained(EXPORT_DIR)
for f in ['configuration_saber.py', 'modeling_saber.py']:
    shutil.copy(REPO_DIR / f, EXPORT_DIR / f)

tokens_b = meta['tokens_seen'] / 1e9
stage_names = {0: 'Foundation', 1: 'Specialization', 2: 'Anneal'}
with open(EXPORT_DIR / 'README.md', 'w') as f:
    f.write(f"""---\nlicense: apache-2.0\ntags: [saber, pretrained, custom-architecture]\n---\n# Eve-3-SABER-0.5B\n500M param decoder-only LM. Tokens: {tokens_b:.1f}B/50B. Stage: {stage_names.get(meta['stage_idx'], '?')}.\n```python\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nmodel = AutoModelForCausalLM.from_pretrained("anthonym21/eve-3-0.5b-saber", trust_remote_code=True)\ntokenizer = AutoTokenizer.from_pretrained("anthonym21/eve-3-0.5b-saber")\n```\n""")

api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(folder_path=str(EXPORT_DIR), repo_id=HF_REPO,
    commit_message=f'Step {meta["step"]:,}, {tokens_b:.1f}B tokens ({stage_names.get(meta["stage_idx"], "?")})')
del mdl
print(f'\n✅ https://huggingface.co/{HF_REPO}')